# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My Rule

For this baseline, I wanted to find pages that are already getting visibility but may need attention.

The rule first checks whether a page has not been updated for a long time and whether it receives enough impressions. These pages are given a higher priority because improving them may increase their performance.

If the page also has a CTR lower than the expected CTR for its position tier, I increase its score even more because it may benefit from content or title improvements.

This baseline is only a simple starting point and is meant to help prioritize pages for manual review.
### Reason Codes

- stale_visible_ctr_gap – The page is stale, receives good impressions, and its CTR is lower than expected.
- stale_but_visible – The page is stale and receives enough impressions but its CTR is acceptable.
- no_action – No strong signal was found by the baseline rule.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## Ranked Queue

After calculating the score for every page, I sorted them from the highest score to the lowest score.

The pages at the top are the ones I would check first because they look like they have the best chance of improving.

The notebook also saves the ranked list as `baseline_action_score.csv`.|

In [18]:
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["has_position"] = df["avg_position"] > 0
df["is_stale"] = df["days_since_last_update"] >= 180
df["is_visible"] = df["impressions_90d"] >= 500

median_ctr_per_tier = df[df["has_position"]].groupby("position_tier")["ctr"].median()
df["expected_ctr"] = df["position_tier"].map(median_ctr_per_tier)
df["ctr_underperforming"] = df["has_position"] & (df["ctr"] < df["expected_ctr"])

df["score"] = 0
mask = df["is_stale"] & df["is_visible"]
df.loc[mask, "score"] = df.loc[mask, "impressions_90d"]
df.loc[mask & df["ctr_underperforming"], "score"] *= 2

def get_reason_code(row):
    if row["is_stale"] and row["is_visible"] and row["ctr_underperforming"]:
        return "stale_visible_ctr_gap"
    elif row["is_stale"] and row["is_visible"]:
        return "stale_but_visible"
    else:
        return "no_action"

df["reason_code"] = df.apply(get_reason_code, axis=1)

df["action"] = "no_action"
df.loc[df["score"] > 0, "action"] = "review_refresh"

ranked = df.sort_values("score", ascending=False).reset_index(drop=True)
ranked["rank"] = ranked.index + 1

out_path = Path("work/outputs/baseline_action_score.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)

cols_to_keep = ["rank", "content_id", "client_id", "score", "reason_code", "action",
                "days_since_last_update", "impressions_90d", "avg_position", "ctr", "position_tier"]
ranked[cols_to_keep].to_csv(out_path, index=False)

print("rows written:", len(ranked))
print()
print(ranked["reason_code"].value_counts())
print()
print(ranked["action"].value_counts())

ranked[cols_to_keep].head(10)

rows written: 30000

reason_code
no_action                29983
stale_but_visible           14
stale_visible_ctr_gap        3
Name: count, dtype: int64

action
no_action         29983
review_refresh       17
Name: count, dtype: int64


,rank,content_id,client_id,score,reason_code,action,days_since_last_update,impressions_90d,avg_position,ctr,position_tier
0,1,content_cf56e2e2e282,client_7f2253d7e2,61678,stale_but_visible,review_refresh,194,61678,19.7,0.15,striking
1,2,content_7368877ea310,client_7f2253d7e2,59472,stale_but_visible,review_refresh,194,59472,24.8,0.13,page_3_5
2,3,content_1bfaa38ff26c,client_7f2253d7e2,25715,stale_but_visible,review_refresh,194,25715,22.2,0.23,page_3_5
3,4,content_5feee3994adb,client_7f2253d7e2,15624,stale_visible_ctr_gap,review_refresh,194,7812,39.0,0.01,page_3_5
4,5,content_0a91db491d14,client_7f2253d7e2,13299,stale_but_visible,review_refresh,193,13299,10.5,0.49,striking
5,6,content_b16bd7307b39,client_7f2253d7e2,9180,stale_visible_ctr_gap,review_refresh,194,4590,31.0,0.00,page_3_5
6,7,content_c2d929d83eaa,client_7f2253d7e2,7558,stale_but_visible,review_refresh,193,7558,17.9,0.20,striking
7,8,content_fe16a55cd13d,client_7f2253d7e2,4556,stale_but_visible,review_refresh,194,4556,16.4,0.33,striking
8,9,content_ecb6215e79fd,client_7f2253d7e2,4429,stale_but_visible,review_refresh,194,4429,25.3,0.38,page_3_5
9,10,content_928af3e22c80,client_7f2253d7e2,1697,stale_but_visible,review_refresh,193,1697,15.8,0.12,striking


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [19]:
ranked[cols_to_keep].head(20)

,rank,content_id,client_id,score,reason_code,action,days_since_last_update,impressions_90d,avg_position,ctr,position_tier
0,1,content_cf56e2e2e282,client_7f2253d7e2,61678,stale_but_visible,review_refresh,194,61678,19.7,0.15,striking
1,2,content_7368877ea310,client_7f2253d7e2,59472,stale_but_visible,review_refresh,194,59472,24.8,0.13,page_3_5
2,3,content_1bfaa38ff26c,client_7f2253d7e2,25715,stale_but_visible,review_refresh,194,25715,22.2,0.23,page_3_5
3,4,content_5feee3994adb,client_7f2253d7e2,15624,stale_visible_ctr_gap,review_refresh,194,7812,39.0,0.01,page_3_5
4,5,content_0a91db491d14,client_7f2253d7e2,13299,stale_but_visible,review_refresh,193,13299,10.5,0.49,striking
5,6,content_b16bd7307b39,client_7f2253d7e2,9180,stale_visible_ctr_gap,review_refresh,194,4590,31.0,0.00,page_3_5
6,7,content_c2d929d83eaa,client_7f2253d7e2,7558,stale_but_visible,review_refresh,193,7558,17.9,0.20,striking
7,8,content_fe16a55cd13d,client_7f2253d7e2,4556,stale_but_visible,review_refresh,194,4556,16.4,0.33,striking
8,9,content_ecb6215e79fd,client_7f2253d7e2,4429,stale_but_visible,review_refresh,194,4429,25.3,0.38,page_3_5
9,10,content_928af3e22c80,client_7f2253d7e2,1697,stale_but_visible,review_refresh,193,1697,15.8,0.12,striking


### Review

Most of the highest-ranked pages are old pages that still receive good impressions. Several also have lower-than-expected CTR, which increases their score.

Because this is only a baseline rule, these pages should be reviewed manually before making changes.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Some pages near the bottom of the ranking have very similar scores, so I am not fully confident about their order.

This rule only uses a few features from the dataset, so it cannot explain everything. Things like seasonality or recent website updates could change the results.

## Leakage Check

I only used the information that was already available in the dataset.

I did not use future data or any labels while creating the score, so I don't think there is any data leakage.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.